In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [ ]:
from langchain.agents import create_agent

system_prompt = """
You are a chef who will receive an image containing the available ingredients and must:
Identify the ingredients present in the image.
Search the web for recipes that include the identified ingredients.
Return a recipe that uses the largest possible number of ingredients identified in the image.
Rules:
You do not need to use all the ingredients; these are simply the ingredients available.
Avoid using ingredients that are not shown in the image. The goal is to prepare a recipe using the available ingredients.
"""

agent = create_agent(
    model='gpt-5-nano',
    tools=[web_search],
    system_prompt=system_prompt
)

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
multimodal_question = HumanMessage(content=[
    # {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)